In [8]:
!pip install cloudscraper


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import requests
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import re

In [24]:
url = "https://nanoreview.net/en/soc-list/rating"
    
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9,vi;q=0.8",
    "Referer": "https://nanoreview.net/"
}


In [27]:
def crawl_soc_ratings(url):

    scraper = cloudscraper.create_scraper(
        browser={
            'browser': 'chrome',
            'platform': 'windows',
            'mobile': False,
        }
    )

    try:
        print("Đang truy cập NanoReview...")
        response = scraper.get(url)
        if response.status_code != 200:
            print(f"Lỗi truy cập: {response.status_code}")
            return None
            
        soup = BeautifulSoup(response.content, "html.parser")
        table = soup.find("table", class_="table-list")
        if not table:
            return None

        rows = table.find("tbody").find_all("tr")
        data_list = []

        for row in rows:
            cols = row.find_all("td")
            if len(cols) >= 3:
                # 1. Rank
                rank = cols[0].get_text(strip=True)

                # 2. Chip Name
                chip_name = cols[1].find("a").get_text(strip=True) if cols[1].find("a") else cols[1].get_text(strip=True)

                # 3. NanoReview Score (Xử lý tách điểm và loại A+, S...)
                # Thường dữ liệu là "97A+" -> dùng regex lấy phần số
                score_raw = cols[2].get_text(separator=" ", strip=True)
                score_match = re.search(r'(\d+)', score_raw)
                nr_score = score_match.group(1) if score_match else ""

                # 4. AnTuTu Score
                # Lấy text và xóa các khoảng trắng hoặc ký tự xuống dòng
                antutu_raw = cols[3].get_text(strip=True) if len(cols) > 3 else ""
                # Giữ lại chỉ các con số
                antutu_score = "".join(re.findall(r'\d+', antutu_raw))

                data_list.append({
                    "Rank": rank,
                    "Chip Name": chip_name,
                    "NanoReview Score": nr_score,
                    "AnTuTu Score": antutu_score
                })

        df = pd.DataFrame(data_list)
        
        # Chuyển kiểu dữ liệu sang số để sau này dễ so sánh
        df["Rank"] = pd.to_numeric(df["Rank"], errors='coerce')
        df["NanoReview Score"] = pd.to_numeric(df["NanoReview Score"], errors='coerce')
        df["AnTuTu Score"] = pd.to_numeric(df["AnTuTu Score"], errors='coerce')
        
        print(f"Đã thu thập {len(df)} chip.")
        return df

    except Exception as e:
        print(f"Lỗi: {e}")
        return None

In [28]:
df_chips = crawl_soc_ratings(url)

if df_chips is not None:
    print(df_chips.head(20)) # Xem 20 chip đầu bảng
    df_chips.to_csv("soc_ratings.csv", index=False, encoding="utf-8-sig")
    print("\nĐã lưu dữ liệu vào file soc_ratings.csv")

Đang truy cập NanoReview...
Đã thu thập 200 chip.
    Rank                   Chip Name  NanoReview Score  AnTuTu Score
0      1    Snapdragon 8 Elite Gen 5                97       3932243
1      2                     A19 Pro                96       2606807
2      3              Dimensity 9500                96       3508681
3      4                 Exynos 2600                93       3145925
4      5                   Apple A19                91       2239708
5      6             Dimensity 9500s                90       3041307
6      7  Snapdragon 8 Elite (Gen 4)                90       3099952
7      8         Dimensity 9400 Plus                89       2874956
8      9              Dimensity 9400                88       2825187
9     10          Snapdragon 8 Gen 5                86       3082388
10    11                    Xring O1                86       2998904
11    12                   Apple A18                82       1918031
12    13                 Exynos 2500                8